# VLM 파싱 모델 비교 (QWEN3 vs OpenAI)



## 1. 구글 드라이버 연동

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

PDF_DIR = "/content/drive/MyDrive/3team_project"
OUTPUT_DIR = "/content/drive/MyDrive/3team_project/vlm_parsed"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"OUTPUT 경로 폴더 생성 : {OUTPUT_DIR}")

OUTPUT 경로 폴더 생성 : /content/drive/MyDrive/3team_project/vlm_parsed


## 2. 패키지 설치

In [4]:
!pip install -q pymupdf Pillow tqdm pandas numpy
!pip install -q openai
!pip install -q transformers>=4.51.0 accelerate qwen-vl-utils
!pip install -q bitsandbytes # INT4 양자화용
!pip install -q langchain langchain-experimental langchain-openai
!pip install -q rouge-score jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 114.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 65

각 패키지의 용도는 다음과 같습니다:

- `pymupdf`, `Pillow`: 문서를 이미지로 변환하고 처리합니다.
(HWP 파일은 보통 PDF로 변환 후, 이 라이브러리들을 통해 이미지로 만들어 AI에게 보여줍니다.)
- `openai`: OpenAI의 GPT-4o(Vision) 모델을 사용하기 위한 라이브러리입니다.
- `transformers`, `qwen-vl-utils`: Qwen2-VL (Qwen3라고 지칭하신 최신 모델 등)을 로컬에서 불러오고 처리하는 데 필요합니다.
- `accelerate`, `bitsandbytes`: Colab GPU 메모리 한계 내에서 거대 모델(Qwen)을 효율적으로 돌리기 위한 최적화(양자화) 도구입니다.
- `langchain` 관련: AI 모델과 데이터를 쉽게 연결해주는 프레임워크입니다.
- `rouge-score`, `jiwer`: 파싱 결과가 얼마나 정확한지 평가(채점)하는 도구입니다.

참고: HWP 파일을 직접 읽는 도구는 포함되어 있지 않습니다.
보통 HWP → PDF 변환 후, PDF를 pymupdf로 이미지화하여 Qwen/OpenAI에게 시각적으로 읽게 하는 방식을 사용합니다.

## 3. 모델 로드

허깅페이스에서 Qwen3-VL-8B 모델을 로드합니다.

In [ ]:
import torch
from transformers import Qwen3